In [1]:
# conda env 'at3d_pure'

from collections import OrderedDict
import at3d
import xarray as xr
import numpy as np

In [2]:
## INPUT
aerosol_file = '../intermediate_data/AEROSOL_MISR_21_0p200.nc'
atmos_file = '../intermediate_data/ATMOS_S07_RTG_1400.nc'

## OUTPUT
out_file = '../intermediate_data/MIXTURE_default.nc'

In [3]:
# wavelength = 0.67

aerosol = xr.load_dataset(aerosol_file)
atmos = xr.load_dataset(atmos_file)

assert np.array_equal(aerosol.x.data, atmos.x.data)
assert np.array_equal(aerosol.y.data, atmos.y.data)
assert np.array_equal(aerosol.z.data, atmos.z.data)

In [4]:
background_optical_scatterers = OrderedDict()
merged_properties = at3d.medium.mix_optical_properties(
    atmos,
    aerosol,
    asymmetry_tol=0.02,
    maxnphase=18
)

ones = np.ones((atmos.dims['x'], atmos.dims['y'], atmos.dims['z']))
if((np.array_equal(atmos.ssalb.data, ones)) & (np.array_equal(aerosol.ssalb.data, ones))):
    merged_properties['ssalb'] = (['x', 'y', 'z'], ones)

merged_properties.to_netcdf(out_file)